# Диффузионные методы переноса стиля

Ноутбук выполняет baseline SD 1.5 + ControlNet Canny + IP-Adapter: подготовку данных, генерацию и расчёт метрик. Теория, анализ результатов и план сравнения с LoRA приведены в [отчёте](https://github.com/Batyaoftheyear/diffusion-style/blob/main/docs/stage1_report.pdf).

В меню **Среда выполнения → Сменить среду выполнения** выберите T4 GPU. Данные и результаты сохраняются в Google Drive и используются при повторном запуске.


## 1. Проверка окружения

Проверяется версия Python и доступность GPU.


In [ ]:
import os
from pathlib import Path
import subprocess
import sys

assert (3, 11) <= sys.version_info[:2] <= (3, 13), "Нужен Python 3.11–3.13 для закреплённого окружения"
subprocess.run(["nvidia-smi"], check=True)
print(sys.version)


## 2. Google Drive

В Drive хранятся данные, кэш моделей и результаты. Код и окружение Python размещаются на временном диске Colab и восстанавливаются после сброса среды.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
HOME_DIR = Path("/content/drive/MyDrive/diffusion_style_transfer")
HOME_DIR.mkdir(parents=True, exist_ok=True)
os.environ["DIFFUSION_STYLE_HOME"] = str(HOME_DIR)
os.environ["HF_HOME"] = str(HOME_DIR / ".cache/huggingface")
os.environ["MPLBACKEND"] = "Agg"
print("Артефакты:", HOME_DIR)


## 3. Загрузка репозитория

По умолчанию загружается ветка `main`. Для запуска конкретной версии задайте её commit SHA в `GIT_REF`. Перед обновлением сохраните изменения в рабочей копии: ячейка обновляет только репозиторий без локальных правок.


In [ ]:
REPO_URL = "https://github.com/Batyaoftheyear/diffusion-style.git"
GIT_REF = "main"
REPO = Path("/content/diffusion-style")


def git_command(*arguments, cwd=None):
    result = subprocess.run(
        ["git", *arguments], cwd=cwd, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
    )
    if result.returncode:
        raise RuntimeError(
            f"Git завершился с кодом {result.returncode}: git {' '.join(arguments)}\n"
            f"{result.stdout.strip()}"
        )
    return result.stdout.strip()


remote_refs = git_command("ls-remote", REPO_URL)
if not remote_refs:
    raise RuntimeError("Удалённый репозиторий не содержит веток и коммитов")

if not REPO.exists():
    print(git_command("clone", REPO_URL, str(REPO)))
assert (REPO / ".git").is_dir(), "Каталог должен быть отдельным git checkout"
remote = git_command("remote", "get-url", "origin", cwd=REPO)
assert remote.removesuffix(".git") == REPO_URL.removesuffix(".git"), "Другой origin"
status = git_command("status", "--porcelain", cwd=REPO)
assert not status, "В checkout есть изменения; сохраните их перед обновлением"
print(git_command("fetch", "origin", GIT_REF, cwd=REPO))
print(git_command("checkout", "--detach", "FETCH_HEAD", cwd=REPO))
print(git_command("rev-parse", "HEAD", cwd=REPO))


## 4. Установка зависимостей

Создаётся отдельное окружение через virtualenv и устанавливаются версии из `requirements.txt`. Контрольная сумма установочного архива проверяется по SHA-256. Если создание окружения было прервано, ячейка восстанавливает pip и продолжает установку.


In [ ]:
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from src.bootstrap import ensure_environment, requirements_fingerprint, run_command

VENV = Path("/content/diffusion-style-env")
PYTHON = ensure_environment(VENV)
requirements = REPO / "requirements.txt"
requirements_sha = requirements_fingerprint(requirements)
marker = VENV / "requirements.sha256"
if not marker.exists() or marker.read_text().strip() != requirements_sha:
    run_command([PYTHON, "-m", "pip", "install", "-r", requirements], cwd=REPO)
    run_command([PYTHON, "-m", "pip", "check"], cwd=REPO)
    marker.write_text(requirements_sha)
else:
    run_command([PYTHON, "-m", "pip", "check"], cwd=REPO)
run_command([PYTHON, "-c", "import torch, diffusers, transformers; print(torch.__version__, diffusers.__version__, transformers.__version__); assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0))"], cwd=REPO)


def run_script(name, *arguments):
    run_command([PYTHON, REPO / "scripts" / name, *arguments], cwd=REPO)


## 5. Проверка кода

Тесты проверяют подготовку данных, отбор изображений, метрики, начальный шум и продолжение прогона. Проверка генерации на GPU выполняется в разделе 8.


In [ ]:
run_command([PYTHON, "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=REPO)


## 6. Подготовка данных

Отбираются 75 картин импрессионизма и 24 фотографии COCO. Пять картин выделяются в референсы, остальные 70 предназначены для обучения LoRA. При повторном запуске проверяются контрольные суммы сохранённых файлов.


In [ ]:
run_script("prepare_data.py")
run_script("prepare_data.py", "--check-only")


## 7. Просмотр данных

Таблица примеров позволяет проверить центральную обрезку, Canny-карты и соответствие текстовых описаний фотографиям. Основной референс выбирается по фиксированному порядку отбора.


In [ ]:
import json
from IPython.display import display, Image as DisplayImage

DATA_DIR = Path(subprocess.check_output([str(PYTHON), "-c", "from src.config import load_config,data_dir; print(data_dir(load_config()))"], cwd=REPO, text=True).strip())
display(DisplayImage(filename=str(DATA_DIR / "preview.png"), width=768))
art = [json.loads(line) for line in (DATA_DIR / "art_manifest.jsonl").read_text().splitlines()]
photos = [json.loads(line) for line in (DATA_DIR / "coco_manifest.jsonl").read_text().splitlines()]
print("Картины:", len(art), "Референсы:", sum(row["split"] == "reference" for row in art), "Фото:", len(photos))
for row in photos[:6]:
    print(row["id"], row["scene_group"], "—", row["prompt"])


## 8. Проверка генерации на двух фото

Загружаются SD 1.5, ControlNet и IP-Adapter из закреплённых версий репозиториев. Пайплайн генерирует два изображения с сохранением начального шума. Эти результаты входят в основной прогон. Ранее полученные изображения используются после проверки целостности.


In [ ]:
run_script("run_baseline.py", "--limit", "2")
RUN_DIR = HOME_DIR / "results" / json.loads((HOME_DIR / "results/current_run.json").read_text())["run_id"]
display(DisplayImage(filename=str(RUN_DIR / "figures/comparisons.png"), width=1000))


## 9. Основной прогон

Оставшиеся фотографии обрабатываются с теми же настройками. Сохранённые результаты проверяются и используются повторно. Одновременно запускать несколько сред с записью в одну папку нельзя.

В прогоне `c66b0a9fe8ce4d583cfd` получено 23 изображения из 24; на `coco_000000423971` сработал safety checker. Дополнительная ячейка ниже отмечает этот случай как пропуск, если остальные результаты сохранены. Пропуск учитывается отдельно от успешных генераций.


In [ ]:
run_script("run_baseline.py")
RUN_DIR = HOME_DIR / "results" / json.loads((HOME_DIR / "results/current_run.json").read_text())["run_id"]
run = json.loads((RUN_DIR / "run.json").read_text())
print("Получено:", run["completed_images"], "; пропущено:", run.get("skipped_images", 0))


In [ ]:
run_script(
    "finalize_run.py",
    "--skip-photo", "coco_000000423971",
    "--reason", "safety_checker",
)


## 10. Метрики и визуализация

После завершения процесса генерации запускается DINOv2. Сводка содержит средние значения и стандартные отклонения по фотографиям при одном seed. Таблица показывает первые шесть сцен фиксированного отбора.


In [17]:
run_script("evaluate.py")
summary = json.loads((RUN_DIR / "summary.json").read_text())
for name, values in summary["metrics"].items():
    print(f"{name}: {values['mean']:.4f} ± {values['std']:.4f}; n={values['n']}")
display(DisplayImage(filename=str(RUN_DIR / "figures/comparisons.png"), width=1000))
print("CSV:", RUN_DIR / "metrics.csv")


/content/diffusion-style-env/lib/python3.13/site-packages/huggingface_hub/file_download.py:801: UserWarning: Not enough free disk space to download the file. The expected file size is: 346.35 MB. The target location /content/drive/MyDrive/diffusion_style_transfer/.cache/huggingface/hub/models--facebook--dinov2-base/blobs only has 289.17 MB free disk space.
  warnings.warn(
Evaluated coco_000000482800
Evaluated coco_000000132622
Evaluated coco_000000076211
Evaluated coco_000000357742
Evaluated coco_000000279730
Evaluated coco_000000236592
Evaluated coco_000000255483
Evaluated coco_000000566042
Evaluated coco_000000044590
Evaluated coco_000000057597
Evaluated coco_000000367095
Evaluated coco_000000021167
Evaluated coco_000000361147
Evaluated coco_000000521231
Evaluated coco_000000017714
Evaluated coco_000000193181
Evaluated coco_000000006771
Evaluated coco_000000094336
Evaluated coco_000000283785
Evaluated coco_000000236730
Evaluated coco_000000503841
Evaluated coco_000000463542
Evaluate

CSV: /content/drive/MyDrive/diffusion_style_transfer/results/c66b0a9fe8ce4d583cfd/metrics.csv


## Результаты baseline

Прогон `c66b0a9fe8ce4d583cfd`: 23 успешных результата, один пропуск. Edge F1 = 0.772 ± 0.149; DINO cosine = 0.482 ± 0.224 при seed 42.

В первых шести примерах сохраняются основные силуэты и расположение объектов, заметны палитра и живописная фактура референса. При этом у людей меняются лица и одежда, а в сценах с сэндвичем и духовкой — фон и предметные детали. Подробный анализ сопоставляет эти наблюдения с метриками.

[Отчёт PDF](https://github.com/Batyaoftheyear/diffusion-style/blob/main/docs/stage1_report.pdf) · [Анализ результатов](https://github.com/Batyaoftheyear/diffusion-style/blob/main/docs/stage1_report.md) · [Первые шесть сцен](https://github.com/Batyaoftheyear/diffusion-style/blob/main/docs/figures/comparisons.png)
